In [ ]:
import importlib, tempfile, collections
import pandas as pd
import scoring, dsh, stage3
importlib.reload(stage3)  
import json

import table_solver as tab, fit_solver as fit, calc_solver as calc
for m in (tab, fit, calc):
    importlib.reload(m)

In [29]:
dev = json.load(open("dev.json"))

In [ ]:
def table_answer(task):
    try:
        return dsh.to_number(tab.solve_data(task))
    except Exception as first:
        try:
            return dsh.to_number(tab.solve_table_semantic(task))
        except Exception as second:
            raise ValueError(f'solve_data: {type(first).__name__}: {first} | semantic: {type(second).__name__}: {second}')

rows = []
for t in dev:
    if not t['files']: 
        continue                               
    payload = scoring.write_files([t], tempfile.mkdtemp())[0]   
    try:
        if t['type'].startswith('fit'):
            answer = stage3.solve_fit_task(payload)
        else:
            answer = dsh.to_number(stage3.solve_data(payload))
        ok = scoring.score_answer(t, answer)[0] == 1
        rows.append((t['id'], t['type'], 'answered', ok, answer, t['answer'], ''))
    except stage3.Unclear as e:
        rows.append((t['id'], t['type'], 'unclear (LLM fit)', False, None, t['answer'], str(e.phrases)))
    except Exception as e:
        rows.append((t['id'], t['type'], 'Error', False, None, t['answer'], f'{type(e).__name__}: {e}'))

res = pd.DataFrame(rows, columns=['id', 'type', 'status', 'right', 'answer', 'gold', 'reason'])
print(res.groupby('type').agg(tasks=('id', 'size'),
                              answered=('status', lambda s: (s == 'answered').sum()),
                              right=('right', 'sum'),
                              unclear=('status', lambda s: (s == 'unclear (LLM fit)').sum())))

rows = []
for t in dev:
    if t['files']:
        continue
    try:
        a = calc.calc_general(t)
        rows.append((t['id'], t['type'], 'answered', scoring.score_answer(t, a)[0] == 1))
    except Exception:
        rows.append((t['id'], t['type'], 'open', False))
c = pd.DataFrame(rows, columns=['id', 'type', 'status', 'right'])
print(c.groupby('type').agg(n=('id', 'size'), answered=('status', lambda s: (s == 'answered').sum()),
                            right=('right', 'sum')))


              tasks  answered  right  unclear
type                                         
fit.linear       18        17     17        1
fit.logistic     17        16     16        1
table.derive     16        15     11        0
table.filter     16         7      3        0
table.group      16        12      6        0
table.join       16         5      1        0
table.stat       16        12     11        0
                   n  answered  right
type                                 
calc.arithmetic    8         5      5
calc.dates         8         8      8
calc.functions     7         6      6
calc.growth        7         7      7
calc.integer       7         7      7
calc.percent       7         7      7
calc.probability   7         7      7
calc.stats_inline  7         7      7
calc.word          7         7      7


In [ ]:
# get all that have returned it wrong and why 
# TODO doesn't work for table and calc yet, need to check
print("CALC")
res[(res.type.str.startswith('calc')) & (~res.right)][['id', 'type', 'reason']]
print("TABLE")
res[(res.type.str.startswith('table')) & (~res.right)][['id', 'type', 'reason']]
print("FIT")
res[(res.type.str.startswith('fit')) & (res.status == 'Error')][['id', 'type', 'reason']]

CALC
TABLE
FIT


,id,type,reason


Print all the wrong answers and that were wrong

In [52]:
res[(res.status == 'answered') & ~res.right][['id', 'type', 'answer', 'gold']]

,id,type,answer,gold
0,d_001,table.filter,27.200000,23.50
15,d_023,table.filter,1700.000000,955.00
17,d_029,table.derive,1.536786,1536.79
20,d_032,table.group,1066.000000,8547.00
27,d_039,table.filter,172.000000,125.70
47,d_070,table.join,921.477273,864.41
49,d_073,table.group,1.000000,11.00
50,d_074,table.filter,1064.411111,953.33
52,d_077,table.derive,20.352564,19.55
61,d_096,table.group,3.500000,62.00


In [ ]:
# if possible, load agent to check agent
importlib.reload(stage3)
agent = stage3.Agent()  

ids = list(res[res.status == 'unclear (LLM fit)'].id)
batch = [t for t in dev if t['id'] in ids]
out = agent.solve(scoring.write_files(batch, tempfile.mkdtemp()), seconds=120)
for t, o in zip(batch, out):
    print(t['id'], '| Answer:', o['answer'], '| gold:', t['answer'],
          '| right' if scoring.score_answer(t, o['answer'])[0] == 1 else '| wrong')

NameError: name 'agent' is not defined

Problem that was solved but to keep it in mind:

In [23]:
# problem r2 is read as RÂ²
t = next(t for t in dev if t['id'] == 'd_012')
print('RÂ²' in t['objective'], 'R²' in t['objective'])

True False
